In [1]:
# Cell 1: Imports and MLflow Setup

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Set experiment
mlflow.set_experiment("Model_Comparison_ROC_AUC")

2026/03/02 07:45:58 INFO mlflow.tracking.fluent: Experiment with name 'Model_Comparison_ROC_AUC' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/unnati/Documents/MLFLOW/mlruns/6', creation_time=1772417758677, experiment_id='6', last_update_time=1772417758677, lifecycle_stage='active', name='Model_Comparison_ROC_AUC', tags={}, workspace='default'>

In [2]:
# Cell 2: Load Dataset and Create Fixed Split

# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# Fixed split for fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42  # fixed seed for fair model comparison
)

print("Dataset ready ✅")
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Dataset ready ✅
Train size: (455, 30)
Test size: (114, 30)


In [3]:
# Cell 3: Define Models to Compare

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    "Logistic_Regression": LogisticRegression(max_iter=500),
    "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=42)
}

print("Models defined ✅")

Models defined ✅


In [4]:
# Cell 4: Train, Evaluate, and Log Each Model

for model_name, model in models.items():
    
    with mlflow.start_run(run_name=model_name):
        
        # Train
        model.fit(X_train, y_train)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        # Log parameters
        mlflow.log_param("model_name", model_name)
        mlflow.log_params(model.get_params())
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        
        # ROC Curve
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.figure()
        plt.plot(fpr, tpr)
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {model_name}")
        plt.savefig("roc_curve.png")
        mlflow.log_artifact("roc_curve.png")
        plt.close()
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)

        plt.figure()
        plt.imshow(cm)
        plt.title(f"Confusion Matrix - {model_name}")
        plt.colorbar()

        plt.xticks([0,1], ["Pred 0", "Pred 1"])
        plt.yticks([0,1], ["Actual 0", "Actual 1"])
        
        # Add numbers inside cells
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, cm[i, j],
                         ha="center", va="center", color="white")
        
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

print("All models logged successfully ✅")

/home/unnati/Documents/MLFLOW/mlflow_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


All models logged successfully ✅


In [5]:
# Cell: Global Explainability Logging

import os

with mlflow.start_run(run_name="Global_Explainability_Analysis"):
    
    explain_models = {
        "Logistic_Regression": LogisticRegression(max_iter=2000),
        "Random_Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient_Boosting": GradientBoostingClassifier(random_state=42)
    }
    
    for model_name, model in explain_models.items():
        model.fit(X_train, y_train)
        
        # -------------------------------
        # Logistic Regression Explainability
        # -------------------------------
        if model_name == "Logistic_Regression":
            
            coefficients = model.coef_[0]
            coef_df = pd.DataFrame({
                "feature": X.columns,
                "coefficient": coefficients
            }).sort_values(by="coefficient", key=abs, ascending=False)
            
            file_name = f"{model_name}_coefficients.csv"
            coef_df.to_csv(file_name, index=False)
            mlflow.log_artifact(file_name)
            
            # Plot Top 10 Coefficients
            top10 = coef_df.head(10)
            plt.figure(figsize=(8,6))
            plt.barh(top10["feature"], top10["coefficient"])
            plt.gca().invert_yaxis()
            plt.title(f"{model_name} - Top 10 Coefficients")
            plot_name = f"{model_name}_coefficients_plot.png"
            plt.savefig(plot_name)
            mlflow.log_artifact(plot_name)
            plt.close()
        
        # -------------------------------
        # Tree-Based Explainability
        # -------------------------------
        else:
            
            importances = model.feature_importances_
            importance_df = pd.DataFrame({
                "feature": X.columns,
                "importance": importances
            }).sort_values(by="importance", ascending=False)
            
            file_name = f"{model_name}_feature_importance.csv"
            importance_df.to_csv(file_name, index=False)
            mlflow.log_artifact(file_name)
            
            # Plot Top 10 Features
            top10 = importance_df.head(10)
            plt.figure(figsize=(8,6))
            plt.barh(top10["feature"], top10["importance"])
            plt.gca().invert_yaxis()
            plt.title(f"{model_name} - Top 10 Feature Importance")
            plot_name = f"{model_name}_importance_plot.png"
            plt.savefig(plot_name)
            mlflow.log_artifact(plot_name)
            plt.close()

print("Global Explainability Logged Successfully ✅")

/home/unnati/Documents/MLFLOW/mlflow_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Global Explainability Logged Successfully ✅


In [6]:
pip install dagshub mlflow --upgrade

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "unnati.c"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "0d3b8060697e2cf9ff4989ed79f3abc9712bdd93"

In [10]:
import dagshub
import mlflow

dagshub.init(repo_owner='unnati.c', repo_name='mlflow-experiments', mlflow=True)

Accessing as unnati.c

Initialized MLflow to track repo "unnati.c/mlflow-experiments"

Repository unnati.c/mlflow-experiments initialized!

In [11]:
mlflow.set_experiment("connection_test")

with mlflow.start_run():
    mlflow.log_param("lr", 0.01)
    mlflow.log_metric("accuracy", 0.95)

2026/03/02 12:08:13 INFO mlflow.tracking.fluent: Experiment with name 'connection_test' does not exist. Creating a new experiment.


🏃 View run thoughtful-newt-461 at: https://dagshub.com/unnati.c/mlflow-experiments.mlflow/#/experiments/0/runs/ec51145209b5433bb538304371c04f5b
🧪 View experiment at: https://dagshub.com/unnati.c/mlflow-experiments.mlflow/#/experiments/0
